# R2-03 · Estadística para modelar — Lección

**Bootcamp de Datos para Funcionarios Públicos · Formación Pública**
Línea R2 · *Científico de Datos* · Semana 5–6 · Se ejecuta en Google Colab.

> 🚀 **Cómo se trabaja:** lee, ejecuta cada celda con **▶︎** (o `Shift`+`Enter`) y completa los `TODO`. Cada ejercicio termina en una **celda de chequeo** que muestra ✅ si está bien o una pista si todavía no.

---

## ¿Qué vas a lograr?

1. Describir una distribución real: centro, cola, **asimetría** y escala log.
2. Detectar **estacionalidad** antes de modelar.
3. Estimar con **intervalos de confianza** (fórmula t y *bootstrap*).
4. Contrastar grupos con una **prueba de hipótesis** e interpretar un p-valor.
5. Medir **correlación** y reconocer una **variable de confusión**.

**Competencia de salida:** usar estadística descriptiva e inferencial para sustentar —y cuestionar— decisiones de modelado.

**Dato real:** PM2.5 diario 2020–2025 de 14 estaciones SINCA (MMA), con temperatura y humedad en Cerro Navia.

**Entregable:** que las **6 celdas de chequeo** muestren ✅.

In [ ]:
# ── Preparación del entorno (ejecuta esta celda primero) ──────────────────────
import os, urllib.request
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# "En vivo o caché": usa el archivo local; si falta (ej. Colab), lo baja del repo.
CSV = "aire_diario.csv"
if not os.path.exists(CSV):
    urllib.request.urlretrieve(f"https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/data/{CSV}", CSV)

df = pd.read_csv(CSV, parse_dates=["fecha"])
df["mes"] = df["fecha"].dt.month
df["anio"] = df["fecha"].dt.year

# Estación ancla: Cerro Navia (Región Metropolitana) — mide PM2.5, temperatura y humedad.
cn = df[df["estacion"] == "Cerro Navia"].copy()
print("Todas las estaciones:", df.shape, "| Cerro Navia:", cn.shape)
df.head()

## El dato: PM2.5 diario en 14 estaciones de Chile

Cada fila es **un día en una estación** del SINCA (Ministerio del Medio Ambiente):
`pm25` es la concentración promedio del día de material particulado fino, en µg/m³.
Algunas estaciones también miden `temp_media` (°C) y `humedad_media` (%).

> 🏛️ **La vara de comparación:** la norma chilena de PM2.5 para 24 horas es **50 µg/m³**
> (D.S. 12/2011 del MMA). La usaremos como referencia a lo largo del módulo.

**¿Por qué estadística antes de modelar?** Porque un modelo hereda todo lo que no viste:
una cola larga, una estacionalidad fuerte, una correlación que en realidad es otra cosa.
En este módulo vas a **describir**, **estimar con incertidumbre**, **contrastar** y
**correlacionar** — y a desconfiar con método.

## 1. Describir antes de modelar: centro, cola y asimetría

El PM2.5 no se reparte como una campana: la mayoría de los días son tranquilos y unos pocos son muy contaminados. Esa **cola larga** se mide con la **asimetría** (*skew*): 0 es simétrico, más de 1 es muy cargado a la derecha. Un truco clásico para domarla es mirar el **logaritmo** (usamos `np.log1p`, que es log(1 + x), porque hay días con PM2.5 = 0).

### ✍️ Ejercicio 1 — Resumen descriptivo y asimetría del PM2.5 (todas las estaciones)

In [ ]:
x = df["pm25"].dropna()
desc = {
    "media":     ...,   # TODO: x.mean()
    "mediana":   ...,   # TODO: x.median()
    "p90":       ...,   # TODO: percentil 90 -> x.quantile(0.90)
    "asimetria": ...,   # TODO: x.skew()
}
# TODO: asimetría del logaritmo -> np.log1p(x).skew()
asim_log = ...
print(desc)
print("asimetría tras log1p:", asim_log)

In [ ]:
# ── Celda de chequeo · Ejercicio 1 ──────────────────────────────────────────
try:
    assert set(desc) == {"media", "mediana", "p90", "asimetria"}
    assert desc["media"] > desc["mediana"], "Con cola a la derecha, la media queda sobre la mediana"
    assert desc["asimetria"] > 1, "El PM2.5 diario es muy asimétrico"
    assert abs(asim_log) < abs(desc["asimetria"]), "El log debería reducir la asimetría"
    print("✅ Ejercicio 1: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'x.mean(), x.median(), x.quantile(0.90), x.skew(); y np.log1p(x).skew() para el logaritmo.')
    print("   Detalle:", e)

In [ ]:
# (ilustración) La misma variable, en escala original y en escala log
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].hist(x, bins=60, color="#4240e5"); ax[0].set_title("PM2.5 diario (µg/m³)")
ax[0].axvline(50, color="#c0392b", ls="--", label="norma 24 h (50)"); ax[0].legend()
ax[1].hist(np.log1p(x), bins=60, color="#ff6b35"); ax[1].set_title("log(1 + PM2.5)")
for a in ax: a.set_ylabel("días")
plt.tight_layout(); plt.show()

## 2. Estacionalidad: el invierno manda

Antes de cualquier modelo, pregunta **cuándo** pasa lo que mides. En Santiago, el PM2.5 sube en otoño-invierno (más calefacción a leña, inversión térmica que atrapa el aire frío y la contaminación cerca del suelo). Trabajamos con la estación ancla, **Cerro Navia** (`cn`).

### ✍️ Ejercicio 2 — Promedio por mes y días sobre la norma en Cerro Navia

In [ ]:
# TODO: PM2.5 promedio por mes (groupby "mes")
por_mes = ...
# TODO: número del mes con mayor promedio (idxmax)
mes_peor = ...
# TODO: cuántos días de Cerro Navia superaron 50 µg/m³
dias_sobre_norma = ...
print("mes con mayor promedio:", mes_peor, "| días sobre 50:", dias_sobre_norma)

In [ ]:
# ── Celda de chequeo · Ejercicio 2 ──────────────────────────────────────────
try:
    assert len(por_mes) == 12
    assert mes_peor in (5, 6, 7, 8), "El peor mes debería caer en otoño-invierno"
    assert dias_sobre_norma == int((cn["pm25"] > 50).sum())
    print("✅ Ejercicio 2: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'cn.groupby("mes")["pm25"].mean(); int(por_mes.idxmax()); int((cn["pm25"] > 50).sum()).')
    print("   Detalle:", e)

In [ ]:
# (ilustración) Distribución diaria por mes: la caja muestra el 50% central de los días
meses = [cn.loc[cn["mes"] == m, "pm25"].dropna() for m in range(1, 13)]
plt.figure(figsize=(9, 4))
plt.boxplot(meses, showfliers=False)
plt.axhline(50, color="#c0392b", ls="--", label="norma 24 h (50 µg/m³)")
plt.xticks(range(1, 13), ["ene", "feb", "mar", "abr", "may", "jun", "jul", "ago", "sep", "oct", "nov", "dic"])
plt.ylabel("PM2.5 diario (µg/m³)"); plt.title("Cerro Navia 2020–2025: PM2.5 diario por mes")
plt.legend(); plt.tight_layout(); plt.show()

## 3. Intervalo de confianza para la media invernal

«El PM2.5 promedio del invierno en Cerro Navia es X» es una **estimación**: con otros inviernos daría otro número. El **intervalo de confianza al 95%** acompaña la cifra con su incertidumbre. Definimos invierno = junio, julio y agosto.

### ✍️ Ejercicio 3 — IC 95% de la media de PM2.5 en invierno

In [ ]:
inv = cn[cn["mes"].isin([6, 7, 8])]["pm25"].dropna()
media_inv = inv.mean()
# TODO: stats.t.interval(0.95, len(inv)-1, loc=media_inv, scale=stats.sem(inv))
ic_low, ic_high = ..., ...
print("media:", round(media_inv, 1), "| IC95%: [", ic_low, ",", ic_high, "]")

In [ ]:
# ── Celda de chequeo · Ejercicio 3 ──────────────────────────────────────────
try:
    assert ic_low < media_inv < ic_high
    assert 0 < ic_high - ic_low < 10, "Con ~550 días el intervalo debería ser angosto (pocos µg/m³)"
    print("✅ Ejercicio 3: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'stats.t.interval(0.95, len(inv)-1, loc=media_inv, scale=stats.sem(inv)) devuelve (bajo, alto).')
    print("   Detalle:", e)

## 4. *Bootstrap*: la diferencia invierno − verano, sin fórmulas

¿Cuánto más contaminado es el invierno que el verano (diciembre a febrero)? El **bootstrap** responde con fuerza bruta: remuestrea los días **con reemplazo** miles de veces, calcula la diferencia de medias en cada remuestra y toma los percentiles 2,5 y 97,5. Sin supuestos de normalidad — útil con colas largas como esta.

### ✍️ Ejercicio 4 — IC 95% por bootstrap de la diferencia de medias

In [ ]:
ver = cn[cn["mes"].isin([12, 1, 2])]["pm25"].dropna()
dif_obs = inv.mean() - ver.mean()
rng = np.random.default_rng(42)
# TODO: 2000 diferencias de medias de remuestras:
#       rng.choice(inv.values, len(inv)).mean() - rng.choice(ver.values, len(ver)).mean()
difs = ...
# TODO: percentiles 2,5 y 97,5 de difs -> np.percentile(difs, [2.5, 97.5])
b_low, b_high = ..., ...
print("diferencia observada:", round(dif_obs, 1), "| IC95% bootstrap:", b_low, b_high)

In [ ]:
# ── Celda de chequeo · Ejercicio 4 ──────────────────────────────────────────
try:
    assert len(difs) == 2000
    assert b_low < dif_obs < b_high
    assert b_low > 0, "El intervalo no toca el 0: el invierno es claramente peor"
    print("✅ Ejercicio 4: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'Lista por comprensión: [rng.choice(inv.values, len(inv)).mean() - rng.choice(ver.values, len(ver)).mean() for _ in range(2000)].')
    print("   Detalle:", e)

In [ ]:
# (ilustración) La distribución bootstrap de la diferencia
try:
    plt.figure(figsize=(6, 3.5))
    plt.hist(difs, bins=40, color="#4240e5")
    for v in (b_low, b_high):
        plt.axvline(v, color="#c0392b", ls="--")
    plt.xlabel("invierno − verano (µg/m³)"); plt.ylabel("remuestras")
    plt.title("2.000 diferencias bootstrap (líneas: IC 95%)"); plt.tight_layout(); plt.show()
except (NameError, TypeError, ValueError):
    print("ℹ️ Completa el ejercicio 4 para ver este gráfico.")

## 5. Prueba de hipótesis: ¿el invierno 2025 fue más limpio que el 2024?

Una autoridad anuncia: «el PM2.5 invernal bajó este año». ¿Es una mejora real o cabe en el azar de un invierno a otro? El **test de Welch** (`stats.ttest_ind(..., equal_var=False)`) compara dos medias: si el *p-valor* < 0,05 diríamos que la diferencia es **significativa**.

> ⚠️ **Letra chica honesta:** los días seguidos se parecen entre sí (no son independientes), así que con series diarias los p-valores tienden a ser **optimistas**. Úsalos como guía, no como veredicto.

### ✍️ Ejercicio 5 — Compara los inviernos 2024 y 2025 en Cerro Navia

In [ ]:
en_invierno = cn["mes"].isin([6, 7, 8])
inv_2024 = cn[en_invierno & (cn["anio"] == 2024)]["pm25"].dropna()
inv_2025 = cn[en_invierno & (cn["anio"] == 2025)]["pm25"].dropna()
# TODO: test de Welch -> stats.ttest_ind(inv_2024, inv_2025, equal_var=False)
t_stat, p_valor = ..., ...
# TODO: True si p_valor < 0.05
significativa = ...
print("media 2024:", round(inv_2024.mean(), 1), "| media 2025:", round(inv_2025.mean(), 1))
print("p-valor:", p_valor, "| ¿significativa al 5%?", significativa)

In [ ]:
# ── Celda de chequeo · Ejercicio 5 ──────────────────────────────────────────
try:
    assert 0.0 <= p_valor <= 1.0
    assert significativa == (p_valor < 0.05)
    assert not significativa, "Con estos datos la baja de 2025 NO es distinguible del azar"
    print("✅ Ejercicio 5: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'stats.ttest_ind(inv_2024, inv_2025, equal_var=False) devuelve (t, p); significativa = p_valor < 0.05.')
    print("   Detalle:", e)

## 6. Correlación ≠ causalidad: la estación del año como confusor

La correlación de Pearson (−1 a 1) mide si dos variables se mueven juntas. En Cerro Navia, más humedad parece ir con más PM2.5. ¿La humedad «causa» contaminación? Mira qué pasa al comparar **solo días de pleno invierno** (junio y julio): si el signo se da vuelta, la relación global era obra de un tercero — **el invierno**, que trae a la vez aire húmedo y más contaminación.

### ✍️ Ejercicio 6 — Correlaciones globales vs dentro del invierno

In [ ]:
s = cn.dropna(subset=["pm25", "temp_media", "humedad_media"])
# TODO: stats.pearsonr(s["temp_media"], s["pm25"]) -> (r, p)
r_temp, p_temp = ..., ...
# TODO: correlación humedad–PM2.5 en todos los días -> s["pm25"].corr(s["humedad_media"])
r_hum_todo = ...
pleno = s[s["mes"].isin([6, 7])]
# TODO: la misma correlación, pero solo en los días de `pleno`
r_hum_invierno = ...
print("r temp:", r_temp, "| r hum (todo):", r_hum_todo, "| r hum (jun-jul):", r_hum_invierno)

In [ ]:
# ── Celda de chequeo · Ejercicio 6 ──────────────────────────────────────────
try:
    assert -1 <= r_temp <= 1 and 0 <= p_temp <= 1
    assert r_temp < 0, "Días más fríos, más PM2.5"
    assert r_hum_todo > 0, "En el año completo, humedad y PM2.5 suben juntas"
    assert r_hum_invierno < 0, "Dentro del invierno el signo se invierte"
    print("✅ Ejercicio 6: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'stats.pearsonr(x, y) devuelve (r, p); para la correlación simple usa serie1.corr(serie2), primero en s y luego en pleno.')
    print("   Detalle:", e)

In [ ]:
# (ilustración) La paradoja en un gráfico: global vs dentro de jun-jul
try:
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
    ax[0].scatter(s["humedad_media"], s["pm25"], s=6, alpha=0.3, color="#4240e5")
    ax[0].set_title(f"Todo el año: r = {r_hum_todo:.2f}")
    ax[1].scatter(pleno["humedad_media"], pleno["pm25"], s=8, alpha=0.5, color="#ff6b35")
    ax[1].set_title(f"Solo junio-julio: r = {r_hum_invierno:.2f}")
    for a in ax: a.set_xlabel("humedad media (%)")
    ax[0].set_ylabel("PM2.5 (µg/m³)"); plt.tight_layout(); plt.show()
except (NameError, TypeError, ValueError):
    print("ℹ️ Completa el ejercicio 6 para ver este gráfico.")

## ✍️ Reflexión (no se corrige, pero es lo más importante)

En el ejercicio 6, la temperatura también correlaciona fuerte con el PM2.5 (r ≈ −0,55): días fríos,
aire sucio. Si un municipio propusiera «subir la temperatura» como política de aire limpio, sería
absurdo — el frío actúa **a través** de la calefacción a leña y de la inversión térmica. En una o dos
frases: ¿qué variable **sí** podría intervenir una política pública, y qué dato te faltaría para medirla?

## Cierre

- **Describe antes de modelar:** el PM2.5 tiene cola larga (media > mediana, asimetría > 1); el log la doma.
- La **estacionalidad** es la primera estructura de cualquier serie pública: el invierno manda.
- Un **intervalo de confianza** (fórmula t o *bootstrap*) acompaña cada cifra con su incertidumbre.
- Una **prueba de hipótesis** separa una mejora real de la variación normal — y un p > 0,05 también es un resultado.
- **Correlación ≠ causalidad:** una tercera variable (la estación del año) puede crear, borrar o invertir una relación.

> *En R2-04 usarás todo esto para construir tu primer modelo: predecir el PM2.5 de mañana.*